# Mindestanforderungen Regression + Evaluation
## Regression
- Entscheiden Sie, ob Sie mit der linearen Regression modellieren, prädizieren, oder beides erreichen wollen.
- Führen Sie die entsprechende(n) Analyse(n) methodisch sauber durch und berichten Sie entsprechend der    eingeführten Kriterien und Evaluationsmaße das Ergebnis.
- Wenn Sie sowohl modellieren als auch prädizieren wollen, führen Sie die Modellierung nur auf den Trainingsdaten durch, um zu vermeiden, dass Sie ungewollt aus Ihren Testdaten lernen. Trainieren Sie das Prädiktionsmodell im zweiten Schritt aufgrund der Erkenntnisse der Modellierung.
## Evaluation
- Definieren Sie für Ihr Modell die Frequenzbaseline bzw. die Mittelwertsbaseline.
- Definieren Sie für Ihr Modell eine einfache Vergleichsbaseline.
- Prüfen Sie mittels einer Lernkurve, ob Ihr Modell zu Over- oder Underfitting neigt und evaluieren Sie entsprechend des Ergebnisses ein mächtigeres oder weniger mächtiges Modell. Wenn Ihr Modell weder Over- noch Underfitting zeigt: Herzlichen Glückwunsch, es ist nichts weiter zu tun.
- Interpretieren Sie Ihr Modell: Entweder mit Hilfe von LIME oder bei transparenten Algorithmen aufgrund des gelernten Modells selber.

## Zielsetzung und methodische Entscheidung

Ziel dieser Analyse ist es, den Zusammenhang zwischen ausgewählten numerischen Merkmalen und der Zielvariable mithilfe einer linearen Regression zu untersuchen.

Dabei werden zwei Zielsetzungen verfolgt:
- Modellierung: Identifikation und Interpretation relevanter Einflussfaktoren auf die Zielvariable.
- Prädiktion: Aufbau eines Regressionsmodells zur Vorhersage der Zielvariable für neue, unbekannte Daten.

Aufgrund dieser Zielsetzung wird zunächst eine modellierende Analyse durchgeführt, bevor im zweiten Schritt ein prädiktives Modell trainiert wird.

Im weiteren Verlauf wird das Modell zudem evaluiert sowie erklärt.

Das Baseline-Modell, welches unter anderem zur Evaluierung beiträgt befindet sich im Notebook `LineareRegression_extended_gridsearch_naive.ipynb`.

# Lineare Regression: Einkommen vorhersagen (`ConvertedTotalComp`)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Laden des Datensatzes und Feature-Selektion

Der Datensatz wird aus einer Datei geladen und in eine Feature-Matrix sowie eine Zielvariable aufgeteilt.
Es werden ausschließlich geeignete numerische Merkmale verwendet, um die Voraussetzungen der linearen Regression zu erfüllen.

Diese Trennung bildet die Basis für alle weiteren Analyseschritte.


In [ ]:

df = pd.read_csv("../One-Hot-Encoded.csv")


q98 = df["ConvertedCompTotal"].dropna().quantile(0.98)
df = df[df["ConvertedCompTotal"] <= q98].copy()
print(df["ConvertedCompTotal"].describe())

df = df[df["ConvertedCompTotal"] > 10000].copy()

bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

## Definition der Zielvariable

Die Zielvariable wird als `ConvertedCompTotal` festgelegt.
Sie repräsentiert das zu modellierende bzw. zu prädizierende Jahreseinkommen und wird in den folgenden Schritten separat behandelt.


In [ ]:
TARGET = "ConvertedCompTotal"

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
feature_cols = [c for c in num_cols if c != TARGET]

X = df[feature_cols].copy()
y = df[TARGET].copy()
X = X.drop(columns=["ConvertedCompYearly"])


# Konsistentes Dropna (Check, dass X vollständig + y vorhanden)
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()

print("After dropna -> X:", X.shape, "y:", y.shape)
y.describe()

In [ ]:
y_model = y.astype(float)
y_model.head()

## Aufteilung in Trainings- und Testdaten

Die Daten werden in Trainings- und Testdaten aufgeteilt.
Die Trainingsdaten werden ausschließlich für die Modellierung und das Lernen von Zusammenhängen verwendet.

Die Testdaten bleiben vollständig unberührt und dienen ausschließlich der späteren Bewertung der Prädiktionsleistung.


In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_model,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## Vorverarbeitung der Features mit `ColumnTransformer`

In dieser Zelle wird die Vorverarbeitung der Feature-Matrix definiert.

Zunächst werden die numerischen Merkmale (`MaxAge`, `AgeNum`, `WorkExp`, `YearsCode`, `RemoteCategoryNum`) explizit festgelegt.  
Alle übrigen Spalten werden als Dummy-Variablen behandelt.

Mithilfe eines `ColumnTransformer` werden unterschiedliche Vorverarbeitungsschritte kombiniert:
- Die numerischen Features werden mit einem `StandardScaler` standardisiert, um vergleichbare Skalen für die lineare Regression zu gewährleisten.
- Die Dummy-Variablen werden unverändert an das Modell weitergegeben (`passthrough`).

Nicht explizit definierte Spalten werden verworfen.  
Diese strukturierte Vorverarbeitung ermöglicht eine saubere Integration in eine Pipeline und verhindert inkonsistente Feature-Skalierungen.


In [ ]:
num_cols = ["AgeNum", "WorkExp", "YearsCode", "RemoteCategoryNum"]
dummy_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("dum", "passthrough", dummy_cols),
    ],
    remainder="drop"
)


## Hyperparameter-Optimierung mit Grid Search und Regularisierung

In dieser Zelle wird eine umfassende Hyperparameter-Optimierung für lineare Regressionsmodelle mit Regularisierung durchgeführt.

Zunächst wird eine Pipeline definiert, die aus zwei Schritten besteht:
- `preprocess`: Anwendung der zuvor definierten Feature-Vorverarbeitung (Skalierung numerischer Merkmale, Weitergabe der Dummy-Variablen)
- `model`: Platzhalter für das jeweilige Regressionsmodell

Im Anschluss wird ein Parametergrid definiert, das drei regularisierte lineare Modelle umfasst:
- Ridge Regression (L2-Regularisierung) mit verschiedenen `alpha`-Werten
- Lasso Regression (L1-Regularisierung) mit variierenden Regularisierungsstärken
- Elastic Net, das L1- und L2-Regularisierung kombiniert, inklusive Variation des Mischparameters `l1_ratio`

Die Modell- und Hyperparameter-Auswahl erfolgt mithilfe einer `GridSearchCV`:
- Bewertet wird das Root Mean Squared Error (RMSE), welches minimiert werden soll
- Die Evaluation erfolgt mittels Cross-Validation auf den Trainingsdaten
- Das beste Modell wird automatisch erneut auf den vollständigen Trainingsdaten trainiert (`refit=True`)

Nach dem Training werden das beste Modell, die optimalen Hyperparameter sowie das zugehörige Cross-Validation-RMSE ausgegeben.
Das beste Pipeline-Modell wird anschließend extrahiert und für die weitere Analyse (Intercept, Koeffizienten) gespeichert.


In [ ]:
# Grid Search (Ridge / Lasso / ElasticNet) mit Preprocessing-Pipeline

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge())  # dient als Platzhalter für die Pipeline. Wird später durch bestes Modell erstetzt
])

param_grid = [
    {
        "model": [Ridge()],
        "model__alpha": [0.01, 0.1, 1, 10, 100, 300, 1000]
    },
    {
        "model": [Lasso(max_iter=20000)],
        "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
    },
    {
        "model": [ElasticNet(max_iter=20000)],
        "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }
]


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",  # Ziel: RMSE minimieren
    cv=3,
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

print("Bestes Modell:", grid.best_estimator_.named_steps["model"])
print("Beste Parameter:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

best_pipe = grid.best_estimator_
clf = best_pipe.named_steps["model"] 


## Lernkurvenanalyse und Diagnose der Modellkomplexität

In dieser Zelle wird eine Lernkurvenanalyse für das zuvor mittels Grid Search ausgewählte beste Modell (`best_pipe`) durchgeführt.

Mithilfe der Funktion `learning_curve` wird untersucht, wie sich der Fehler (RMSE) in Abhängigkeit von der Anzahl der Trainingsbeispiele entwickelt:
- Der Train-RMSE beschreibt die Anpassung des Modells an die Trainingsdaten.
- Der Cross-Validation-RMSE gibt Aufschluss über die Generalisierungsfähigkeit.

Die Lernkurve wird für mehrere Trainingsgrößen berechnet und grafisch dargestellt. Zusätzlich werden die Standardabweichungen visualisiert, um die Stabilität der Schätzungen zu beurteilen.

Im Anschluss erfolgt eine heuristische Diagnose:
- Der Unterschied zwischen Trainings- und Validierungsfehler (Generalization Gap) wird berechnet.
- Dieser Gap wird relativ zum Validierungsfehler bewertet.
- Als grober Referenzwert dient die RMSE eines Mean-Predictors (Standardabweichung der Zielvariable im Training).

Auf Basis dieser Kennzahlen wird das Modell qualitativ als Underfitting, Overfitting oder angemessen angepasst klassifiziert.  
Diese Analyse ergänzt die numerischen Evaluationsmaße um eine strukturelle Beurteilung der Modellkomplexität und des Lernverhaltens.


In [ ]:
train_sizes, train_scores, cv_scores = learning_curve(
    estimator=best_pipe,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

# In positiven RMSE umwandeln
train_rmse = -train_scores
cv_rmse = -cv_scores

train_rmse_mean = train_rmse.mean(axis=1)
train_rmse_std  = train_rmse.std(axis=1)
cv_rmse_mean    = cv_rmse.mean(axis=1)
cv_rmse_std     = cv_rmse.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_rmse_mean, marker="o", label="Train RMSE")
plt.fill_between(train_sizes, train_rmse_mean-train_rmse_std, train_rmse_mean+train_rmse_std, alpha=0.15)

plt.plot(train_sizes, cv_rmse_mean, marker="o", label="CV RMSE")
plt.fill_between(train_sizes, cv_rmse_mean-cv_rmse_std, cv_rmse_mean+cv_rmse_std, alpha=0.15)

plt.title("Lernkurve (RMSE) für best_pipe")
plt.xlabel("Anzahl Trainingsbeispiele")
plt.ylabel("RMSE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


gap = cv_rmse_mean[-1] - train_rmse_mean[-1]
gap_ratio = gap / max(cv_rmse_mean[-1], 1e-12)

baseline_train_rmse = float(np.std(y_train, ddof=0))

print(f"Letzter Punkt: Train RMSE={train_rmse_mean[-1]:.4f}, CV RMSE={cv_rmse_mean[-1]:.4f}")
print(f"Generalization Gap (CV-Train)={gap:.4f}  | Gap-Ratio={gap_ratio:.2%}")
print(f"Train-Baseline (Mean-Predictor) RMSE ~ std(y_train)={baseline_train_rmse:.4f}")


## Bewertung von Over- und Underfitting mittels Lernkurve

Anhand der Lernkurve wurde untersucht, ob das Modell zu Over- oder Underfitting neigt.
Dabei wurden Trainings- und Cross-Validation-RMSE in Abhängigkeit von der Trainingsdatenmenge betrachtet.

Die Lernkurve zeigt, dass sich Trainings- und Validierungsfehler bei wachsender Datenmenge annähern.
Der Generalization Gap bleibt moderat, während beide Fehler auf einem stabilen Niveau konvergieren.

Dies deutet darauf hin, dass weder starkes Overfitting noch Underfitting vorliegt.
Ein mächtigeres Modell (z. B. geringere Regularisierung oder nichtlineare Erweiterungen) ist daher nicht zwingend erforderlich.
Ebenso besteht kein Bedarf, die Modellkomplexität weiter zu reduzieren.

Das aktuell gewählte regularisierte lineare Modell stellt einen angemessenen Kompromiss zwischen Bias und Varianz dar.


## Analyse und Interpretation der Modellkoeffizienten

In dieser Zelle werden die Koeffizienten des besten linearen Modells aus der Grid Search analysiert, um die modellierten Zusammenhänge inhaltlich zu interpretieren.

Zunächst werden das trainierte Regressionsmodell (`model`) sowie die zugehörige Vorverarbeitung (`preprocess`) aus der Pipeline extrahiert.
Anschließend werden die Feature-Namen direkt aus dem `ColumnTransformer` übernommen. Falls dies aufgrund der verwendeten scikit-learn-Version nicht möglich ist, wird ein robuster Fallback verwendet, der numerische und Dummy-Features explizit kennzeichnet.

Sofern das gewählte Modell Koeffizienten besitzt (z. B. Ridge, Lasso oder ElasticNet), werden diese:
- den zugehörigen Features zugeordnet,
- nach ihrem absoluten Betrag sortiert,
- und tabellarisch ausgegeben.

Zusätzlich werden die stärksten positiven und negativen Einflüsse separat dargestellt.
Der Intercept wird ebenfalls ausgegeben und beschreibt den erwarteten Zielwert bei Feature-Werten von Null.

Diese Auswertung dient der Modellierung im Sinne der Aufgabenstellung, da sie Einblicke in Richtung und Stärke der Zusammenhänge zwischen Merkmalen und Zielvariable liefert.
Sollte das gewählte Modell keine expliziten Koeffizienten besitzen, wird auf alternative Interpretationsmethoden hingewiesen.


In [ ]:
model = best_pipe.named_steps["model"]
prep = best_pipe.named_steps["preprocess"]

try:
    feature_names = prep.get_feature_names_out()
except Exception:
    num_cols = prep.transformers_[0][2]
    dum_cols = prep.transformers_[1][2]
    feature_names = np.array([f"num__{c}" for c in num_cols] + [f"dum__{c}" for c in dum_cols])

# Koeffizienten prüfen
if hasattr(model, "coef_"):
    coefs = model.coef_.ravel()
    coef_series = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)

    print("Intercept:", getattr(model, "intercept_", None))
    display(coef_series.head(20).to_frame("coef (abs-sortiert)"))

    # Top 10 positive Einflüsse
    print("\nTop positive Einflüsse:")
    display(coef_series.sort_values(ascending=False).head(10).to_frame("coef"))

    # Top 10 negative Einflüsse
    print("\nTop negative Einflüsse:")
    display(coef_series.sort_values(ascending=True).head(10).to_frame("coef"))
else:
    print("Das aktuelle Modell hat keine coef_. Verwende stattdessen LIME oder Permutation Importance.")


## Prädiktion auf den Testdaten und finale Modellbewertung

In dieser Zelle wird das zuvor mittels Grid Search ausgewählte beste Modell (`best_pipe`) zur Vorhersage auf den Testdaten eingesetzt.

Zunächst werden die Prädiktionen für die Test-Feature-Matrix erzeugt und den tatsächlichen Zielwerten gegenübergestellt.
Als Referenz wird zusätzlich eine Baseline berechnet, bei der für alle Testbeobachtungen der Mittelwert der Zielvariable aus den Trainingsdaten vorhergesagt wird.

Die Modellleistung wird anhand mehrerer etablierter Evaluationsmaße beurteilt:
- RMSE (Root Mean Squared Error) zur Bewertung der durchschnittlichen quadratischen Abweichung
- R² zur Einschätzung des erklärten Varianzanteils
- MAE (Mean Absolute Error) als robusteres Maß für den mittleren absoluten Fehler

Durch den Vergleich mit der Baseline kann beurteilt werden, ob das Regressionsmodell einen substanziellen Mehrwert gegenüber einer trivialen Vorhersage liefert.
Diese Evaluation erfolgt ausschließlich auf den Testdaten und stellt damit eine unverzerrte Bewertung der Prädiktionsleistung dar.


In [ ]:
# Prediktion mit bestem Modell aus Grid Search

predicted = best_pipe.predict(X_test)
expected = y_test

# Mittelwertsbaseline mit ausgeben für Modellevaluation
test_predict_mean = np.full(len(y_test), y_train.mean())
print("Baseline Mean RMSE:")
rmse_baseline = np.sqrt(mean_squared_error(expected, test_predict_mean))
print(rmse_baseline)

print("RMSE:")
rmse = np.sqrt(mean_squared_error(expected, predicted))
print(rmse)

print("R^2:")
r2 = r2_score(expected, predicted)
print(r2)

print("MAE:")
mae = mean_absolute_error(expected, predicted)
print(mae)


## Übersicht der besten Cross-Validation-Ergebnisse

In dieser optionalen Zelle werden die Ergebnisse der Grid Search aus der Cross-Validation detailliert ausgewertet.

Die vollständigen Cross-Validation-Ergebnisse (`cv_results_`) werden in ein DataFrame überführt und auf die relevantesten Informationen reduziert:
- Rang des Modells innerhalb der Grid Search
- Mittlerer Validierungsfehler
- Streuung der Validierungsergebnisse
- Zugehörige Hyperparameter-Kombinationen

Da in der Grid Search das negative RMSE als Optimierungskriterium verwendet wurde, wird der Wert wieder in ein positives RMSE zurückgerechnet, um die Interpretation zu erleichtern.

Die Tabelle zeigt die besten Modellkonfigurationen nach RMSE sortiert und ermöglicht einen transparenten Vergleich der getesteten Regularisierungsansätze und Hyperparameter.
Diese Auswertung unterstützt die Nachvollziehbarkeit der Modellselektion.


In [ ]:
# Top-Ergebnisse aus der GridSearch nach RMSE sortiert

cv_res = pd.DataFrame(grid.cv_results_)
cols_show = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
cv_res_sorted = cv_res[cols_show].sort_values("rank_test_score").head(10)

# RMSE aus neg_root_mean_squared_error zurückrechnen
cv_res_sorted = cv_res_sorted.assign(
    mean_RMSE = -cv_res_sorted["mean_test_score"],
    std_RMSE = cv_res_sorted["std_test_score"]
).drop(columns=["mean_test_score", "std_test_score"])

cv_res_sorted


## Vergleich von tatsächlichen und vorhergesagten Zielwerten

In dieser Zelle wird die Vorhersagequalität des Modells grafisch beurteilt.
Dazu werden die tatsächlichen Zielwerte den vorhergesagten Werten in einem Streudiagramm gegenübergestellt.

Die rote Diagonale repräsentiert die ideale Vorhersage, bei der tatsächlicher und vorhergesagter Wert übereinstimmen.
Punkte nahe dieser Linie deuten auf eine gute Modellanpassung hin, während systematische Abweichungen auf Bias oder Heteroskedastizität hinweisen können.

Diese Visualisierung ergänzt die numerischen Evaluationsmaße und erlaubt eine intuitive Einschätzung der Prädiktionsgüte über den gesamten Wertebereich der Zielvariable hinweg.


In [ ]:
plt.figure()
plt.scatter(expected, predicted, alpha=0.3)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red", linewidth=2
)
plt.xlabel("Actual Income")
plt.ylabel("Predicted Income")
plt.title("Predicted vs. Actual Income")
plt.show()



## Residuenanalyse zur Modellvalidierung

In dieser Zelle werden die Residuen des Regressionsmodells analysiert.
Die Residuen ergeben sich als Differenz zwischen den tatsächlichen und den vorhergesagten Zielwerten.

Im Streudiagramm werden die Residuen in Abhängigkeit von den vorhergesagten Werten dargestellt.
Die horizontale Linie bei Null kennzeichnet eine fehlerfreie Vorhersage.

Eine zufällige Streuung der Punkte um die Nulllinie spricht für ein gut angepasstes Modell.
Systematische Muster oder eine fächerförmige Struktur können hingegen auf Modellverletzungen wie Nichtlinearität oder Heteroskedastizität hinweisen.

Diese Residuenanalyse dient der qualitativen Überprüfung der Modellannahmen der linearen Regression.


In [ ]:
residuals = expected - predicted

plt.figure()
plt.scatter(predicted, residuals, alpha=0.3)
plt.axhline(0)
plt.xlabel("Predicted Income")
plt.ylabel("Residuals")
plt.title("Residuals vs. Predicted")
plt.show()


## Vergleich der Verteilungen von tatsächlichen und vorhergesagten Werten

In dieser Zelle werden die Verteilungen der tatsächlichen Zielwerte und der vom Modell vorhergesagten Werte gegenübergestellt.
Beide Verteilungen werden in einem Histogramm visualisiert, um Unterschiede in Lage, Streuung und Form zu erkennen.

Ein ähnlicher Verlauf der beiden Verteilungen deutet darauf hin, dass das Modell die grundlegende Struktur der Zielvariable gut erfasst.
Abweichungen können auf systematische Über- oder Unterschätzungen in bestimmten Wertebereichen hinweisen.

Diese Analyse ergänzt die Punkt- und Residuenplots und unterstützt die Gesamtbewertung der Prädiktionsqualität des Modells.


In [ ]:
plt.figure()
plt.hist(expected, bins=50, alpha=0.5, label="Actual")
plt.hist(predicted, bins=50, alpha=0.5, label="Predicted")
plt.legend()
plt.title("Distribution: Actual vs. Predicted Income")
plt.show()


## Erweiterte Residuen- und Fit-Analyse

In dieser Zelle werden zwei ergänzende Visualisierungen zur Bewertung des Regressionsmodells erstellt.

Im ersten Plot wird die Verteilung der Residuen dargestellt.  
Die Residuen ergeben sich als Differenz zwischen tatsächlichen und vorhergesagten Werten. Eine symmetrische Verteilung um Null spricht für ein unverzerrtes Modell, während Schiefe oder Mehrgipfligkeit auf systematische Fehler hinweisen können. Die eingezeichnete Nulllinie dient als Referenz.

Der zweite Plot zeigt den Regression Fit in Form eines Streudiagramms der tatsächlichen gegen die vorhergesagten Werte.  
Die gestrichelte Diagonale repräsentiert die ideale Vorhersage. Je näher die Punkte an dieser Linie liegen, desto besser ist die Anpassung des Modells.

Diese beiden Visualisierungen ergänzen die numerischen Evaluationsmaße und ermöglichen eine visuelle Beurteilung von Fehlerstruktur und Vorhersagequalität.


In [ ]:
residuals = expected - predicted

plt.figure(figsize=(12,5))

# Residuals Verteilung
plt.subplot(1,2,1)
sns.histplot(residuals, bins=30, kde=True, color="blue")
plt.axvline(x=0, color='red', linestyle='--')
plt.title("Residuals Distribution")
plt.xlabel("Residuals (y_actual - y_predicted)")
plt.ylabel("Frequency")

# Regression Fit
plt.subplot(1,2,2)
sns.scatterplot(x=expected, y=predicted, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')  # Perfect fit line
plt.title("Regression Fit: Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.tight_layout()
plt.show()

## Fazit

Im Rahmen dieser Analyse wurde entschieden, sowohl eine modellierende als auch eine prädiktive lineare Regression durchzuführen. 
Die Modellierung diente der Identifikation und Interpretation relevanter Einflussfaktoren auf das Einkommen, während die Prädiktion auf eine möglichst genaue Vorhersage für neue Daten abzielte.

Die linearen Regressionsmodelle wurden methodisch sauber umgesetzt:  
Die Daten wurden in Trainings- und Testdaten aufgeteilt, eine strukturierte Vorverarbeitung mittels `ColumnTransformer` integriert und regularisierte Regressionsverfahren (Ridge, Lasso, ElasticNet) mithilfe einer Pipeline und Grid Search systematisch verglichen. 
Die Modellselektion erfolgte ausschließlich auf Basis der Trainingsdaten, wodurch eine Verzerrung durch die Testdaten vermieden wurde.

Die Evaluation auf den Testdaten zeigt eine solide Modellleistung.  
Mit einem R²-Wert von 0,59 kann das Modell rund 60 % der Varianz der Zielvariable erklären. 
Der RMSE von 38 341 verdeutlicht die durchschnittliche Abweichung der Vorhersagen vom tatsächlichen Einkommen und liefert eine gut interpretierbare Größenordnung des Vorhersagefehlers.

Zusätzliche Analysen mittels Lernkurven, Residuenplots und Verteilungsvergleichen deuten darauf hin, dass weder starkes Overfitting noch Underfitting vorliegt und das Modell stabil generalisiert.

Insgesamt wurden die gesetzten Ziele erreicht:  
Das finale Modell liefert sowohl interpretierbare Zusammenhänge zwischen den Merkmalen und der Zielvariable als auch eine belastbare Prädiktionsleistung.
Damit sind die Mindestanforderungen der Aufgabenstellung vollständig erfüllt.
